# Supplementary Figure S4 — Per-cell operator agreement on real data (S11)

Sources: `benchmarks/real_flow/operator_agreement.csv` (λ₁ Spearman and leading-eigenvector |cos| between
operators) and `benchmarks/real_flow/null.csv` (five permutation nulls per comparison block).

In [1]:
from _style import *

prov = Provenance("FigS4", notebook="notebooks/supp_s11_operator_agreement.ipynb", kind="supplementary")
oa = pd.read_csv(prov.use("benchmarks/real_flow/operator_agreement.csv", "a", "S11"))
nu = pd.read_csv(prov.use("benchmarks/real_flow/null.csv", "a", "S11"))
V = prov.value

def null_for(r):
    if r.comparison_kind == "independent_sources":
        k = "independent_sources"; m = (nu.left == r.left) & (nu.right == r.right)
    elif r.comparison_kind == "estimators":
        k = "estimator_" + r.right_estimator; m = (nu.left == r.left) & (nu.right == r.right)
    else:
        return None
    n = nu[(nu.dataset == r.dataset) & (nu.null_kind == k) & m]
    return n if len(n) else None

rows = oa.copy()
rows["label"] = rows.apply(lambda r: f"{r.dataset}: {r.left} vs {r.right} ({r.right_estimator})", axis=1)
fig, axs = plt.subplots(1, 2, figsize=(DOUBLE_COL, 4.4), sharey=True, gridspec_kw={"wspace": 0.1})
fig.subplots_adjust(left=0.37, right=0.99, bottom=0.17, top=0.97)
kcol = {"independent_sources": C_FLOW, "estimators": OKABE["green"], "seed_variability": OKABE["purple"]}
for ax, metric, lab in [(axs[0], "lambda1_spearman", "λ₁ Spearman"), (axs[1], "leading_eigenvector_abs_cosine_median", "median |cos| leading eigenvector")]:
    for i, (_, r) in enumerate(rows.iterrows()):
        ax.scatter(r[metric], i, color=kcol[r.comparison_kind], s=14, zorder=3)
        n = null_for(r)
        if n is not None:
            ax.scatter(n[metric], np.full(len(n), i), color=C_BASE, s=5, marker="|", zorder=2)
            V(f"{r.label}|{metric}|null_max", float(n[metric].max()))
        V(f"{r.label}|{metric}", float(r[metric]))
    ax.axvline(0, color=C_BASE, lw=0.5); ax.set_xlabel(lab)
axs[0].set_yticks(range(len(rows))); axs[0].set_yticklabels(rows.label, fontsize=6); axs[0].invert_yaxis()
axs[1].legend(handles=[plt.Line2D([], [], marker="o", ls="", color=c, ms=3.5, label=k.replace("_", " ")) for k, c in kcol.items()]
              + [plt.Line2D([], [], marker="|", ls="", color=C_BASE, ms=5, label="permutation nulls")], fontsize=6, loc="upper center", bbox_to_anchor=(0.45, -0.11), ncol=2, handletextpad=0.2, columnspacing=1.0)
panel_letter(axs[0], "a"); panel_letter(axs[1], "b")
prov.panel("a", "S11 per-cell operator agreement (lambda1 Spearman, eigenvector |cos|) for independent sources, estimators and fit seeds, with permutation nulls")
save(fig, "FigS4", kind="supplementary")

[save] supplementary/FigS4: 181.5 x 108.8 mm, PNG 4288 x 2570 px @ 600 dpi


(PosixPath('/Users/terooatt/Documents/scJDO2-figures/manuscript/figures/supplementary/FigS4.pdf'),
 PosixPath('/Users/terooatt/Documents/scJDO2-figures/manuscript/figures/supplementary/FigS4.png'))

In [2]:
v = prov.values
k = "larry: ot vs lineage (measured_s0)"
prov.legend(f"""
**Supplementary Figure S4 | Per-cell flow operators on real data agree across fit seeds and estimators but not across independent velocity sources (S11, pre-registered;
6 of 14 criteria met).** LARRY (OT vs lineage velocity, PCA-30, 4,000 cells/day) and scNT-seq haematopoiesis (labelling, scVelo, dynamo velocities). For each
comparison: Spearman of per-cell λ₁ (**a**) and median |cos| of the leading eigenvectors (**b**); grey ticks: five permutation nulls. LARRY OT vs lineage:
λ₁ Spearman {f2(v[k + '|lambda1_spearman'])} (null max {f2(v[k + '|lambda1_spearman|null_max'])}), |cos| {f2(v[k + '|leading_eigenvector_abs_cosine_median'])}
(null max {f2(v[k + '|leading_eigenvector_abs_cosine_median|null_max'])}). Seed-variability rows have no null (report only).
""")
prov.write()